# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

The Content Opportunity Scoring task is formulated as a **supervised classification and ranking problem**:
1. **Binary Classification Stage:** We estimate the posterior probability $P(\text{is\_declining} = 1 \mid X)$, where $X$ is a feature vector capturing historical search impressions, rank position, CTR, user engagement, and content age.
2. **Priority Ranking Stage:** We combine model-predicted decline probability with business visibility demand (log-impressions) and CTR efficiency gaps into a composite opportunity score:
   $$\text{Refresh Score} = w_1 \cdot P(\text{decay}) + w_2 \cdot \text{log\_impressions} + w_3 \cdot \text{ctr\_gap}$$
   This ranks candidate pages so editorial reviewers focus their limited bandwidth on high-visibility assets experiencing genuine decay.

In [1]:
import pandas as pd
import numpy as np

data_path = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print(f"Total candidate rows for ML task: {len(df):,}")
print("Task formulation: Binary Classification (Decay Risk) + Learning to Rank (Editorial Priority)")

Total candidate rows for ML task: 30,000
Task formulation: Binary Classification (Decay Risk) + Learning to Rank (Editorial Priority)


## 2. Target or proxy

**Target Definition:**
- **Variable:** `is_declining_label \in \{0, 1\}`
- **Operational Definition:** Defined as `1` if `trend_direction == 'down'`, and `0` if `trend_direction \in {'up', 'flat'}`.
- **Business Alignment:** A downward trend over the 90-day window indicates loss of search visibility and clicks relative to prior historical periods.
- **Leakage Prevention Rule:** `trend_direction` and `trend_pct` are **strictly excluded** from the feature matrix $X$ because they are mathematical formulations of the label.

In [2]:
# Define target label explicitly
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
positive_count = df['is_declining_label'].sum()
total_count = len(df)
base_rate = positive_count / total_count

print(f"Target variable: 'is_declining_label'")
print(f"Positive count:  {positive_count:,}")
print(f"Negative count:  {total_count - positive_count:,}")
print(f"Target Base Rate: {base_rate:.4f} ({base_rate*100:.2f}%)")

Target variable: 'is_declining_label'
Positive count:  16,262
Negative count:  13,738
Target Base Rate: 0.5421 (54.21%)


## 3. Success metric

In operational editorial workflows, teams review a bounded batch of recommendations every sprint (e.g. 20, 50, or 100 pages). Therefore, standard accuracy is uninformative due to base rate skew (54.2%).

**Primary Metric:**
- **Precision@50:** The proportion of truly declining pages among the top 50 highest-ranked recommendations.
- **Precision@100:** Top-100 precision to test sustained ranking quality.

**Secondary Metrics:**
- **ROC-AUC & Average Precision (PR-AUC):** Global discrimination across all threshold settings, especially PR-AUC which accounts for class imbalance.
- **Lift over Baseline:** $\text{Lift} = \frac{\text{Precision@50}_{\text{Model}}}{\text{Precision@50}_{\text{Baseline}}}$. Our goal is $> 2.0\times$ lift.

In [3]:
def precision_at_k(y_true, y_scores, k=50):
    idx = np.argsort(y_scores)[::-1][:k]
    return np.mean(np.array(y_true)[idx])

# Demonstrate metric behavior on random guess vs perfect ranking
np.random.seed(42)
dummy_scores = np.random.rand(len(df))
print(f"Simulated Random Guess Precision@50:  {precision_at_k(df['is_declining_label'], dummy_scores, k=50):.2f} (matches base rate ~{base_rate:.2f})")

Simulated Random Guess Precision@50:  0.56 (matches base rate ~0.54)


## 4. The unit of analysis, as a real dataframe

The unit of analysis is a single content page (`content_id`) within an enterprise client (`client_id`) over the 90-day snapshot.
Below is a sample of the core observation unit showing traffic, ranking, and engagement features:

In [4]:
core_cols = [
    'content_id', 'client_id', 'content_type', 'impressions_90d',
    'clicks_90d', 'avg_position', 'ctr', 'content_age_days',
    'engagement_rate', 'is_declining_label'
]
sample_df = df[core_cols].head(5)
print("Unit of analysis representation:")
print(sample_df.to_string())

Unit of analysis representation:
             content_id          client_id     content_type  impressions_90d  clicks_90d  avg_position   ctr  content_age_days  engagement_rate  is_declining_label
0  content_304f48230142  client_f369cb89fc  keyword article             3803          29          10.6  0.76               187             5.88                   1
1  content_a1fb4e703a9e  client_4e07408562  keyword article            15320           7          20.3  0.05               445             0.00                   1
2  content_9aa793d4d895  client_7f2253d7e2  keyword article            12581          11          36.5  0.09               141             0.00                   1
3  content_331d6c4de07b  client_19581e27de  keyword article            11751          58           6.2  0.49               463             1.28                   0
4  content_d99b7a2d90ca  client_3fdba35f04  keyword article            19140          24          44.0  0.13               263             0.00    

## 5. Why ML beats a fixed rule here

A fixed heuristic rule (e.g., *Flag if impressions > 500 AND avg_position < 20 AND ctr < median*) fails in real search environments for three reasons:
1. **Complex Nonlinear Interactions:** An article ranking in position 8 with a 1.2% CTR might actually be outperforming expectations for its query intent, while an article in position 2 with 3.5% CTR is severely decaying. Static thresholds cannot adapt.
2. **Format and Category Heterogeneity:** B2B technical documentation naturally has lower search volume and different scroll behaviors than consumer blog posts. Fixed rules either swamp editors with technical docs or miss decaying flagship guides.
3. **Multi-Signal Trade-offs:** Machine learning models (e.g. Random Forest) combine 18+ numerical and categorical signals simultaneously, calibrating decay probability against content age and velocity.

In [5]:
# Heuristic rule test
naive_rule = (df['impressions_90d'] > df['impressions_90d'].median()) & (df['avg_position'] < 20)
heuristic_precision = df.loc[naive_rule, 'is_declining_label'].mean()
print(f"Naive heuristic rule flag count: {naive_rule.sum():,} pages")
print(f"Naive heuristic precision:     {heuristic_precision:.3f} (hardly better than the {base_rate:.3f} base rate!)")

Naive heuristic rule flag count: 10,903 pages
Naive heuristic precision:     0.589 (hardly better than the 0.542 base rate!)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.